# Case4 decision — pilot or scale

Scaffold for Case 4: decision tree with EVSI, two-period real-option lattice, FMEA table, and a governance cost sheet.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ri128-rutgers/quality-management-fall2026/blob/main/case4_decision.ipynb) · Also opens in Azure ML Studio notebooks or any Jupyter; nothing here needs a GPU.

**How to use:** run every cell top to bottom once. Cells marked `# >>> YOUR WORK` are where you change inputs or add interpretation. Keep your AI-assistant transcript if you used one; the verification log asks what you checked.

In [ ]:
# Setup — works in Colab and locally
import os, sys, io, urllib.request
REPO = 'ri128-rutgers/quality-management-fall2026'
DATA_BASE = f'https://raw.githubusercontent.com/{REPO}/main/' if 'google.colab' in sys.modules or not os.path.exists('../data') else '../data/'
if 'google.colab' in sys.modules:
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/{REPO}/main/tvm.py', 'tvm.py')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tvm import *
pd.options.display.float_format = '{:,.2f}'.format
def load(name): return pd.read_csv(DATA_BASE + name)
print('data from', DATA_BASE)

In [ ]:
A = load('case4_data_assumptions.csv').set_index('parameter')['value']; rel = load('case4_pilot_reliability.csv'); fh = load('case4_action_failure_history.csv')
states = ['high','medium','low']; prior = np.array([float(A['adoption_high_prob']), float(A['adoption_medium_prob']), float(A['adoption_low_prob'])])
benefit = np.array([float(A['net_benefit_5yr_high_usd']), float(A['net_benefit_5yr_medium_usd']), float(A['net_benefit_5yr_low_usd'])])
deploy = float(A['fleet_deployment_cost_usd']); pilot_cost = float(A['pilot_cost_usd'])
emv_deploy = float(prior @ benefit) - deploy
print('EMV deploy now:', round(emv_deploy), ' | do nothing: 0')

In [ ]:
# Pilot as information: Bayes over the reliability matrix, then EVSI
L = rel[['pilot_says_high','pilot_says_medium','pilot_says_low']].to_numpy()   # rows: true state; cols: pilot signal
p_signal = prior @ L
emv_after = 0
for j, sig in enumerate(['high','medium','low']):
    post = prior*L[:,j]/p_signal[j]
    best = max(float(post @ benefit) - deploy, 0.0)
    print(f'pilot says {sig:6s}: P={p_signal[j]:.3f}, posterior={post.round(2)}, best action value={best:,.0f}')
    emv_after += p_signal[j]*best
EVSI = emv_after - max(emv_deploy, 0)
print('EMV with pilot (before pilot cost):', round(emv_after), ' EVSI:', round(EVSI), ' pilot worth it?', EVSI > pilot_cost)
EVPI = float((prior*np.maximum(benefit-deploy, 0)).sum()) - max(emv_deploy, 0); print('EVPI:', round(EVPI))

In [ ]:
# Risk aversion: exponential utility with the COO's risk tolerance
R = float(A['risk_tolerance_usd']); u = lambda x: 1-np.exp(-x/R); ce = lambda eu: -R*np.log(1-eu)
print('certainty equivalent of deploy-now:', round(ce(float(prior @ u(benefit-deploy)))))
# >>> YOUR WORK: one sentence — does risk aversion change the recommendation?

In [ ]:
# Two-period binomial lattice: option to abandon after the pilot (and to expand at year 2)
V0 = float(prior @ benefit)          # value of the deployed program's benefits
sigma = np.std(np.log(np.maximum(benefit, 1))) # >>> YOUR WORK: justify a volatility from the scenarios
u_, d_ = np.exp(sigma), np.exp(-sigma); rf = 0.04; p = (np.exp(rf)-d_)/(u_-d_)
expand_cost = float(A['expansion_option_cost_usd']); expand_mult = 1.35   # >>> YOUR WORK
nodes = {(2,k): V0*u_**k*d_**(2-k) for k in range(3)}
val2 = {k: max(v - deploy, (v*expand_mult - deploy - expand_cost), 0) for k, v in nodes.items()}
val1 = {k: np.exp(-rf)*(p*val2[(2,k+1)] + (1-p)*val2[(2,k)]) for k in range(2)}
val0 = np.exp(-rf)*(p*val1[1] + (1-p)*val1[0])
print('sigma', round(sigma,3), ' option-inclusive program value after pilot:', round(val0), ' vs EMV deploy-now:', round(emv_deploy))

In [ ]:
# FMEA for an agent that acts: fill five rows; RPN and expected annual loss side by side
actions_yr = float(A['agent_actions_per_day_fleet'])*365
fmea = pd.DataFrame([
  # failure mode, severity 1-10, occurrence (prob per action), detection 1-10 (10 = undetectable), $ per event
  ['stale availability data -> double-booked dock', 6, 0.0025, 5, 900],
  ['misread exception -> wrong re-plan', 5, 0.0018, 6, 400],
  ['tool timeout retry loop -> repeated writes', 4, 0.0012, 3, 150],
  ['wrong dock type -> equipment mismatch', 7, 0.0006, 7, 2_400],
  ['unintended action outside policy', 9, 0.0002, 9, 12_000],   # >>> YOUR WORK: replace with your own five modes
], columns=['mode','S','p_per_action','D','cost_per_event'])
fmea['O'] = pd.cut(fmea.p_per_action, [0, 1e-4, 5e-4, 1e-3, 2e-3, 1], labels=[2,4,6,8,10]).astype(int)
fmea['RPN'] = fmea.S*fmea.O*fmea.D
fmea['expected_annual_loss'] = actions_yr*fmea.p_per_action*fmea.cost_per_event
print(fmea.sort_values('RPN', ascending=False).round(0))
print('observed wrong-action rate from history:', round(fh.wrong_actions.sum()/fh.actions.sum(), 4))

In [ ]:
# Governance cost sheet: map recurring assurance activities to NIST AI RMF functions
gov = pd.DataFrame([
  ['Govern', 'AI policy, roles, quarterly review board', 0.15*195_000],
  ['Map', 'Use-case and impact inventory refresh', 20_000],
  ['Measure', 'Evaluation harness runs + red-team (2/yr)', 90_000],
  ['Measure', 'Human review sampling of actions', actions_yr*0.05*float(A['human_checkpoint_cost_per_review_usd'])],
  ['Manage', 'Logging/audit platform share', float(A['logging_audit_platform_usd_per_year'])],
  ['Manage', 'Incident response + model documentation upkeep', 35_000],
], columns=['rmf_function','activity','annual_usd'])
print(gov.round(0)); print('total annual governance cost:', round(gov.annual_usd.sum()))
# >>> YOUR WORK: ISO/IEC 42001 — add the certification line and argue for or against in cost terms